# Compute Pairwise EI Network

? notebook ?? `loc_model_stage2/stage2_real_fmri_macro/model_scale3.pkl` ? `loc_data_real_fmri/generated_data.npz` ?? real-fMRI ??????????

????????????? EI ????????????????????????????? `loc_result_stage2/stage2_real_fmri_macro`?


## Plan

- ? `summary_scale3.csv` ?? stage2 ????????
- ?? `Parellel_Renorm_Dynamic` ??? `model_scale3.pkl` ? `state_dict`?
- ??????????????????? `sigmas_matrix`?
- ?? `src/models_macro.py` ?? `compute_pairwise_ei_network` ??????? EI ???
- ?????????????????????????


In [ ]:
from __future__ import annotations

import random
import sys
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from torch import nn

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "models_macro.py").exists() and (candidate / "loc_model_stage2").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root containing src/models_macro.py and loc_model_stage2.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models_macro import (  # noqa: E402
    Parellel_Renorm_Dynamic,
    _build_windows_from_series,
    compute_pairwise_ei_network,
)

print(f"project_root: {PROJECT_ROOT}")


project_root: E:\code\Infer-Effective-Connection


In [ ]:
# Configuration
RUN_NAME = "stage2_real_fmri_macro"
MODEL_SCALE = 1
DATA_PATH = PROJECT_ROOT / "loc_data_real_fmri" / "generated_data.npz"
MODEL_PATH = PROJECT_ROOT / "loc_model_stage2" / RUN_NAME / f"model_scale{MODEL_SCALE}.pkl"
SUMMARY_PATH = PROJECT_ROOT / "loc_result_stage2" / RUN_NAME / f"summary_scale{MODEL_SCALE}.csv"
OUTPUT_DIR = PROJECT_ROOT / "loc_result_stage2" / RUN_NAME

# EI sampling follows the training-time function default. Increase for smoother estimates.
EI_SAMPLES = 1000
L = 1.0
DEVICE = None  # None: auto-select cuda:0 when available, otherwise cpu.
MAX_WINDOWS_PER_SUBJECT = None  # None uses all windows from each subject.


def resolve_device(requested_device: str | None = None) -> torch.device:
    if requested_device:
        return torch.device(requested_device)
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    return torch.device("cpu")


DEVICE = resolve_device(DEVICE)
print(f"device: {DEVICE}")
print(f"data_path: {DATA_PATH}")
print(f"model_path: {MODEL_PATH}")
print(f"summary_path: {SUMMARY_PATH}")


device: cpu
data_path: E:\code\Infer-Effective-Connection\loc_data_real_fmri\generated_data.npz
model_path: E:\code\Infer-Effective-Connection\loc_model_stage2\stage2_real_fmri_macro\model_scale3.pkl
summary_path: E:\code\Infer-Effective-Connection\loc_result_stage2\stage2_real_fmri_macro\summary_scale3.csv


## Helpers

?? helper ? `compute_jacobian_matrix.py` ? stage2 artifact ???????????? `models_macro.py` ?? EI ???????


In [3]:
def parse_int_list(value: Any) -> List[int]:
    if value is None:
        return []
    if isinstance(value, float) and np.isnan(value):
        return []
    text = str(value).strip()
    if text == "" or text.lower() == "nan":
        return []
    return [int(part.strip()) for part in text.split(",") if part.strip()]


def load_stage2_artifact(summary_path: Path, model_path: Path, model_scale: int) -> Dict[str, Any]:
    if not model_path.exists():
        raise FileNotFoundError(f"Missing model checkpoint: {model_path}")
    if not summary_path.exists():
        raise FileNotFoundError(f"Missing summary file: {summary_path}")

    summary_row = pd.read_csv(summary_path).iloc[0].to_dict()
    scale_dims = parse_int_list(summary_row.get("scale_dims", ""))
    reduce_dims = parse_int_list(summary_row.get("reduce_dims", ""))
    group = parse_int_list(summary_row.get("group", ""))
    logical_scale_id = int(summary_row.get("scale_id", max(int(model_scale) - 1, 0)))

    group_schedule = str(summary_row.get("group_schedule", "")).strip()
    if group_schedule and group_schedule.lower() != "nan":
        raise NotImplementedError(
            "This notebook reconstructs the real-fMRI macro model from group/reduce_dims. "
            "Non-empty group_schedule reconstruction is not implemented here."
        )

    return {
        "model_scale": int(model_scale),
        "logical_scale_id": logical_scale_id,
        "scale_dims": scale_dims,
        "reduce_dims": reduce_dims,
        "group": group,
        "hidden_units1": int(summary_row.get("hidden_units1", 64)),
        "hidden_units2": int(summary_row.get("hidden_units2", 64)),
        "flow_num_layers": int(summary_row.get("flow_num_layers", 3)),
        "dynamics_num_layers": int(summary_row.get("dynamics_num_layers", 4)),
        "latent_size": int(summary_row.get("latent_size", 1)),
        "time_delay": int(summary_row.get("time_delay", 3)),
        "encoder_type": str(summary_row.get("encoder_type", "mlp")),
        "model_path": model_path,
        "summary_path": summary_path,
    }


def load_real_fmri_archive(data_path: Path) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    if not data_path.exists():
        raise FileNotFoundError(f"Missing data file: {data_path}")
    with np.load(data_path, allow_pickle=False) as archive:
        data = np.asarray(archive["data"], dtype=np.float32)
        group = np.asarray(archive["group"], dtype=np.int64) if "group" in archive else np.array([], dtype=np.int64)
        if "subject_ids" in archive:
            subject_ids = np.asarray(archive["subject_ids"]).astype(str)
        else:
            subject_ids = np.array([f"subject_{idx + 1}" for idx in range(data.shape[0] if data.ndim == 3 else 1)])
    return data, group, subject_ids


def build_state_model(num_nodes: int, artifact: Dict[str, Any], group_from_data: np.ndarray, device: torch.device) -> Parellel_Renorm_Dynamic:
    group = artifact["group"] or [int(value) for value in np.asarray(group_from_data).reshape(-1).tolist()]
    model = Parellel_Renorm_Dynamic(
        sym_size=int(num_nodes),
        latent_size=int(artifact["latent_size"]),
        effect_size=int(num_nodes),
        cut_size=2,
        hidden_units1=int(artifact["hidden_units1"]),
        hidden_units2=int(artifact["hidden_units2"]),
        normalized_state=True,
        device=device,
        is_random=False,
        flow_num_layers=int(artifact["flow_num_layers"]),
        dynamics_num_layers=int(artifact["dynamics_num_layers"]),
        decode_noise_scale=0.0,
        reduce_dims=artifact["reduce_dims"] or None,
        group=group or None,
        encoder_type=artifact["encoder_type"],
    ).to(device)

    if artifact["scale_dims"] and list(model.scale_dims) != list(artifact["scale_dims"]):
        raise ValueError(f"Rebuilt scale_dims={model.scale_dims} does not match summary scale_dims={artifact['scale_dims']}.")

    state_dict = torch.load(artifact["model_path"], map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model


In [4]:
def maybe_subsample_windows(
    x_windows: np.ndarray,
    y_windows: np.ndarray,
    max_windows: int | None,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    total = int(len(x_windows))
    if max_windows is None or total <= int(max_windows):
        indices = np.arange(total, dtype=np.int64)
        return x_windows, y_windows, indices

    rng = np.random.default_rng(int(seed))
    indices = np.sort(rng.choice(total, size=int(max_windows), replace=False)).astype(np.int64)
    return x_windows[indices], y_windows[indices], indices


def compute_subject_pairwise_ei_network(
    model: Parellel_Renorm_Dynamic,
    subject_series: np.ndarray,
    subject_id: str,
    subject_index: int,
    artifact: Dict[str, Any],
    device: torch.device,
    ei_samples: int,
    L: float,
    max_windows_per_subject: int | None = None,
) -> Dict[str, Any]:
    scale_id = int(artifact["logical_scale_id"])
    time_delay = int(artifact["time_delay"])

    x_windows, y_windows = _build_windows_from_series(subject_series, time_delay)
    if len(x_windows) == 0:
        raise ValueError(f"No usable windows for subject {subject_id}.")
    x_windows, y_windows, used_indices = maybe_subsample_windows(
        x_windows,
        y_windows,
        max_windows=max_windows_per_subject,
        seed=SEED + int(subject_index),
    )

    x_tensor = torch.as_tensor(x_windows, dtype=torch.float32, device=device)
    y_tensor = torch.as_tensor(y_windows, dtype=torch.float32, device=device)
    mse_raw = nn.MSELoss(reduction="none")

    with torch.no_grad():
        sigmas, sigmas_matrix, _, _ = model.estimate_sigmas_matrix(
            x_tensor,
            y_tensor,
            scale_id=scale_id,
            mse_raw=mse_raw,
        )

    torch.manual_seed(SEED + int(subject_index))
    if device.type == "cuda":
        torch.cuda.manual_seed_all(SEED + int(subject_index))

    ei_causal_graph, mean_jacobian, expected_log_abs_J, variance_term = compute_pairwise_ei_network(
        model,
        sigmas_matrix,
        scale_id=scale_id,
        L=float(L),
        device=device,
        num_samples=int(ei_samples),
    )

    return {
        "subject_id": str(subject_id),
        "subject_index": int(subject_index),
        "window_count": int(len(x_windows)),
        "used_window_indices": used_indices,
        "sigmas": sigmas.detach().cpu().numpy().astype(np.float32),
        "sigmas_matrix": sigmas_matrix.detach().cpu().numpy().astype(np.float32),
        "mean_jacobian": mean_jacobian.detach().cpu().numpy().astype(np.float32),
        "jacobian_mean_abs": mean_jacobian.detach().abs().cpu().numpy().astype(np.float32),
        "ei_causal_graph": ei_causal_graph.detach().cpu().numpy().astype(np.float32),
        "ei_expected_log_abs_J": expected_log_abs_J.detach().cpu().numpy().astype(np.float32),
        "ei_variance_term": variance_term.detach().cpu().numpy().astype(np.float32),
    }


def compute_pairwise_ei_network_for_real_fmri(
    model: Parellel_Renorm_Dynamic,
    data: np.ndarray,
    subject_ids: np.ndarray,
    artifact: Dict[str, Any],
    device: torch.device,
    ei_samples: int = 1000,
    L: float = 1.0,
    max_windows_per_subject: int | None = None,
) -> List[Dict[str, Any]]:
    if data.ndim == 2:
        iterable = [(0, str(subject_ids[0] if len(subject_ids) else "subject_1"), data)]
    elif data.ndim == 3:
        iterable = [(idx, str(subject_ids[idx] if idx < len(subject_ids) else f"subject_{idx + 1}"), data[idx]) for idx in range(data.shape[0])]
    else:
        raise ValueError(f"Expected data shape [subjects, time, features] or [time, features], got {data.shape}.")

    results = []
    for subject_index, subject_id, subject_series in iterable:
        result = compute_subject_pairwise_ei_network(
            model=model,
            subject_series=subject_series,
            subject_id=subject_id,
            subject_index=subject_index,
            artifact=artifact,
            device=device,
            ei_samples=ei_samples,
            L=L,
            max_windows_per_subject=max_windows_per_subject,
        )
        results.append(result)
        print(
            f"subject={subject_id} windows={result['window_count']} "
            f"ei_mean={result['ei_causal_graph'].mean():.6f} "
            f"sigma_mean={result['sigmas'].mean():.6f}"
        )
    return results


In [5]:
def stack_result(results: List[Dict[str, Any]], key: str) -> np.ndarray:
    return np.stack([np.asarray(result[key], dtype=np.float32) for result in results], axis=0)


def average_subject_results(results: List[Dict[str, Any]]) -> Dict[str, np.ndarray]:
    matrix_keys = [
        "sigmas_matrix",
        "mean_jacobian",
        "jacobian_mean_abs",
        "ei_causal_graph",
        "ei_expected_log_abs_J",
        "ei_variance_term",
    ]
    averages = {key: stack_result(results, key).mean(axis=0).astype(np.float32) for key in matrix_keys}
    averages["sigmas"] = stack_result(results, "sigmas").mean(axis=0).astype(np.float32)
    return averages


def save_matrix_csv(path: Path, values: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    values = np.asarray(values, dtype=np.float32)
    pd.DataFrame(values).to_csv(path, index=False)


def save_pairwise_ei_outputs(
    output_dir: Path,
    artifact: Dict[str, Any],
    subject_ids: np.ndarray,
    results: List[Dict[str, Any]],
    averages: Dict[str, np.ndarray],
    data_path: Path,
    device: torch.device,
    ei_samples: int,
    L: float,
) -> Dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    model_scale = int(artifact["model_scale"])

    output_paths = {
        "jacobian_mean_abs": output_dir / f"jacobian_mean_abs_scale{model_scale}.csv",
        "sigmas": output_dir / f"sigmas_scale{model_scale}.csv",
        "sigmas_matrix": output_dir / f"sigmas_matrix_scale{model_scale}.csv",
        "mean_jacobian": output_dir / f"mean_jacobian_scale{model_scale}.csv",
        "ei_causal_graph": output_dir / f"ei_causal_graph_scale{model_scale}.csv",
        "ei_expected_log_abs_J": output_dir / f"ei_expected_log_abs_J_scale{model_scale}.csv",
        "ei_variance_term": output_dir / f"ei_variance_term_scale{model_scale}.csv",
        "subject_npz": output_dir / f"pairwise_ei_subjects_scale{model_scale}.npz",
        "subject_summary": output_dir / f"pairwise_ei_subject_summary_scale{model_scale}.csv",
        "run_summary": output_dir / f"pairwise_ei_summary_scale{model_scale}.csv",
    }

    for key in [
        "jacobian_mean_abs",
        "sigmas",
        "sigmas_matrix",
        "mean_jacobian",
        "ei_causal_graph",
        "ei_expected_log_abs_J",
        "ei_variance_term",
    ]:
        save_matrix_csv(output_paths[key], averages[key])

    np.savez_compressed(
        output_paths["subject_npz"],
        subject_ids=np.asarray(subject_ids).astype(str),
        sigmas=stack_result(results, "sigmas"),
        sigmas_matrix=stack_result(results, "sigmas_matrix"),
        mean_jacobian=stack_result(results, "mean_jacobian"),
        jacobian_mean_abs=stack_result(results, "jacobian_mean_abs"),
        ei_causal_graph=stack_result(results, "ei_causal_graph"),
        ei_expected_log_abs_J=stack_result(results, "ei_expected_log_abs_J"),
        ei_variance_term=stack_result(results, "ei_variance_term"),
    )

    subject_summary = pd.DataFrame(
        [
            {
                "subject_id": result["subject_id"],
                "subject_index": result["subject_index"],
                "window_count": result["window_count"],
                "scale_dim": int(result["ei_causal_graph"].shape[0]),
                "ei_mean": float(result["ei_causal_graph"].mean()),
                "ei_std": float(result["ei_causal_graph"].std()),
                "sigma_mean": float(result["sigmas"].mean()),
                "jacobian_mean_abs_mean": float(result["jacobian_mean_abs"].mean()),
            }
            for result in results
        ]
    )
    subject_summary.to_csv(output_paths["subject_summary"], index=False)

    pd.DataFrame(
        [
            {
                "run_name": RUN_NAME,
                "model_scale": model_scale,
                "logical_scale_id": int(artifact["logical_scale_id"]),
                "scale_dim": int(averages["ei_causal_graph"].shape[0]),
                "subject_count": int(len(results)),
                "ei_samples": int(ei_samples),
                "L": float(L),
                "time_delay": int(artifact["time_delay"]),
                "device": str(device),
                "data_path": str(data_path.resolve()),
                "model_path": str(Path(artifact["model_path"]).resolve()),
                "summary_source_path": str(Path(artifact["summary_path"]).resolve()),
                "ei_causal_graph_path": str(output_paths["ei_causal_graph"].resolve()),
            }
        ]
    ).to_csv(output_paths["run_summary"], index=False)

    return output_paths


## Run inference

???????????????????????? real-fMRI ???????? 10 ????


In [6]:
artifact = load_stage2_artifact(SUMMARY_PATH, MODEL_PATH, MODEL_SCALE)
data, group_from_data, subject_ids = load_real_fmri_archive(DATA_PATH)
model = build_state_model(
    num_nodes=int(data.shape[-1]),
    artifact=artifact,
    group_from_data=group_from_data,
    device=DEVICE,
)

print(f"data_shape: {data.shape}")
print(f"subject_count: {len(subject_ids)}")
print(f"group: {group_from_data.tolist()}")
print(f"scale_dims: {model.scale_dims}; logical_scale_id: {artifact['logical_scale_id']}")

subject_results = compute_pairwise_ei_network_for_real_fmri(
    model=model,
    data=data,
    subject_ids=subject_ids,
    artifact=artifact,
    device=DEVICE,
    ei_samples=EI_SAMPLES,
    L=L,
    max_windows_per_subject=MAX_WINDOWS_PER_SUBJECT,
)

averages = average_subject_results(subject_results)
output_paths = save_pairwise_ei_outputs(
    output_dir=OUTPUT_DIR,
    artifact=artifact,
    subject_ids=subject_ids,
    results=subject_results,
    averages=averages,
    data_path=DATA_PATH,
    device=DEVICE,
    ei_samples=EI_SAMPLES,
    L=L,
)

print("saved outputs:")
for name, path in output_paths.items():
    print(f"  {name}: {path}")


data_shape: (10, 895, 1258)
subject_count: 10
group: [116, 167, 183, 171, 191, 188, 242]
scale_dims: [7, 3, 1]; logical_scale_id: 2
subject=A00028185 windows=892 ei_mean=2.732618 sigma_mean=0.000228
subject=A00033747 windows=892 ei_mean=2.587240 sigma_mean=0.000305
subject=A00035072 windows=892 ei_mean=2.478129 sigma_mean=0.000379
subject=A00035827 windows=892 ei_mean=2.715078 sigma_mean=0.000236
subject=A00035840 windows=892 ei_mean=2.354582 sigma_mean=0.000490
subject=A00037112 windows=892 ei_mean=2.152997 sigma_mean=0.000730
subject=A00037511 windows=892 ei_mean=2.716869 sigma_mean=0.000236
subject=A00038998 windows=892 ei_mean=2.337446 sigma_mean=0.000502
subject=A00039391 windows=892 ei_mean=2.229339 sigma_mean=0.000625
subject=A00039431 windows=892 ei_mean=2.713702 sigma_mean=0.000237
saved outputs:
  jacobian_mean_abs: E:\code\Infer-Effective-Connection\loc_result_stage2\stage2_real_fmri_macro\jacobian_mean_abs_scale3.csv
  sigmas: E:\code\Infer-Effective-Connection\loc_result_s

## Result preview

?????????????????????????????????


In [7]:
print("Average EI causal graph:")
display(pd.DataFrame(averages["ei_causal_graph"]))

print("Per-subject summary:")
display(pd.read_csv(output_paths["subject_summary"]))


Average EI causal graph:


,0
0,2.5018


Per-subject summary:


,subject_id,subject_index,window_count,scale_dim,ei_mean,ei_std,sigma_mean,jacobian_mean_abs_mean
0,A00028185,0,892,1,2.732618,0.0,0.000228,0.960260
1,A00033747,1,892,1,2.587240,0.0,0.000305,0.961482
2,A00035072,2,892,1,2.478129,0.0,0.000379,0.961378
3,A00035827,3,892,1,2.715078,0.0,0.000236,0.960704
4,A00035840,4,892,1,2.354582,0.0,0.000490,0.965326
5,A00037112,5,892,1,2.152997,0.0,0.000730,0.963182
6,A00037511,6,892,1,2.716869,0.0,0.000236,0.962524
7,A00038998,7,892,1,2.337446,0.0,0.000502,0.960460
8,A00039391,8,892,1,2.229339,0.0,0.000625,0.962067
9,A00039431,9,892,1,2.713702,0.0,0.000237,0.961196
